## 🎯 Objectives

The workflow in this notebook includes:

- Loading processed training and testing datasets
- Training multiple classification models
- Evaluating model performance
- Comparing all trained models
- Selecting the best-performing model
- Saving the trained model for deployment

## 📈 Expected Outcome

By the end of this notebook, we will have a production-ready machine learning model capable of predicting loan default risk with high accuracy and reliability.

In [3]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib
import warnings

warnings.filterwarnings("ignore")

# 📥 Load Processed Dataset

In [4]:
X_train = pd.read_csv("../data/processed/X_train.csv")

X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv")

y_test = pd.read_csv("../data/processed/y_test.csv")

In [5]:
print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Target :", y_train.shape)

print("Testing Target :", y_test.shape)

Training Features : (204277, 24)
Testing Features : (51070, 24)
Training Target : (204277, 1)
Testing Target : (51070, 1)


In [6]:
results = []

def evaluate_model(model, model_name):

    predictions = model.predict(X_test)

    probability = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(y_test, predictions)

    recall = recall_score(y_test, predictions)

    f1 = f1_score(y_test, predictions)

    roc_auc = roc_auc_score(y_test, probability)

    print("="*60)

    print(model_name)

    print("="*60)

    print(f"Accuracy  : {accuracy:.4f}")

    print(f"Precision : {precision:.4f}")

    print(f"Recall    : {recall:.4f}")

    print(f"F1 Score  : {f1:.4f}")

    print(f"ROC AUC   : {roc_auc:.4f}")

    print()

    print("Confusion Matrix")

    print(confusion_matrix(y_test, predictions))

    print()

    print(classification_report(y_test, predictions))

    results.append([
        model_name,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ])

# Logistic Regression

In [7]:
# Logistic Regression

logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logistic_model.fit(X_train, y_train.values.ravel())

evaluate_model(logistic_model, "Logistic Regression")

Logistic Regression
Accuracy  : 0.8852
Precision : 0.6024
Recall    : 0.0332
F1 Score  : 0.0630
ROC AUC   : 0.7531

Confusion Matrix
[[45009   130]
 [ 5734   197]]

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.60      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.74      0.52      0.50     51070
weighted avg       0.85      0.89      0.84     51070



## Business Insight

Logistic Regression acts as the benchmark model.

Its performance helps us understand whether more complex models such as Random Forest and XGBoost provide meaningful improvements.

# 🌳 Decision Tree

Decision Trees learn decision rules by recursively splitting the data based on feature values.

They can capture nonlinear relationships and are highly interpretable.

In [9]:
decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree.fit(
    X_train,
    y_train.values.ravel()
)

evaluate_model(
    decision_tree,
    "Decision Tree"
)

Decision Tree
Accuracy  : 0.8014
Precision : 0.1958
Recall    : 0.2285
F1 Score  : 0.2108
ROC AUC   : 0.5526

Confusion Matrix
[[39572  5567]
 [ 4576  1355]]

              precision    recall  f1-score   support

           0       0.90      0.88      0.89     45139
           1       0.20      0.23      0.21      5931

    accuracy                           0.80     51070
   macro avg       0.55      0.55      0.55     51070
weighted avg       0.81      0.80      0.81     51070



# 🌲 Random Forest

Random Forest combines multiple decision trees to improve predictive performance and reduce overfitting.

It is one of the most widely used ensemble learning algorithms for classification tasks.

In [10]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train.values.ravel()
)

evaluate_model(
    random_forest,
    "Random Forest"
)

Random Forest
Accuracy  : 0.8853
Precision : 0.6437
Recall    : 0.0283
F1 Score  : 0.0543
ROC AUC   : 0.7412

Confusion Matrix
[[45046    93]
 [ 5763   168]]

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.64      0.03      0.05      5931

    accuracy                           0.89     51070
   macro avg       0.77      0.51      0.50     51070
weighted avg       0.86      0.89      0.84     51070



# ⚡ XGBoost

Extreme Gradient Boosting (XGBoost) is a powerful boosting algorithm designed to achieve high predictive performance.

It sequentially builds trees by correcting the errors of previous trees.

In [11]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train.values.ravel()
)

evaluate_model(
    xgb_model,
    "XGBoost"
)

XGBoost
Accuracy  : 0.8865
Precision : 0.5928
Recall    : 0.0727
F1 Score  : 0.1295
ROC AUC   : 0.7544

Confusion Matrix
[[44843   296]
 [ 5500   431]]

              precision    recall  f1-score   support

           0       0.89      0.99      0.94     45139
           1       0.59      0.07      0.13      5931

    accuracy                           0.89     51070
   macro avg       0.74      0.53      0.53     51070
weighted avg       0.86      0.89      0.85     51070



## Business Insight

XGBoost is widely used in banking, fintech, and credit risk applications because of its ability to model complex relationships while maintaining high predictive performance.

# 📊 Model Comparison

After training all models, their performance metrics are compared to identify the best-performing classifier.


In [12]:
comparison = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
)

comparison.sort_values(
    by="ROC-AUC",
    ascending=False,
    inplace=True
)

comparison.reset_index(
    drop=True,
    inplace=True
)

comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,XGBoost,0.886509,0.592847,0.072669,0.129468,0.754396
1,Logistic Regression,0.885177,0.602446,0.033215,0.062959,0.753111
2,Random Forest,0.885334,0.643678,0.028326,0.054264,0.741198
3,Decision Tree,0.801390,0.195753,0.228461,0.210846,0.552565


In [13]:
comparison.style.highlight_max(
    subset=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    color="lightgreen"
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,XGBoost,0.886509,0.592847,0.072669,0.129468,0.754396
1,Logistic Regression,0.885177,0.602446,0.033215,0.062959,0.753111
2,Random Forest,0.885334,0.643678,0.028326,0.054264,0.741198
3,Decision Tree,0.801390,0.195753,0.228461,0.210846,0.552565


## Business Insight

The comparison table summarizes the strengths of each algorithm.

The selected model should not only achieve high accuracy but also maintain strong recall and ROC-AUC, which are particularly important for identifying high-risk borrowers.

In [14]:
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    xgb_model,
    "../models/final_model.pkl"
)

print("Best model saved successfully.")

Best model saved successfully.


# 📝 Conclusion

In this notebook, multiple supervised machine learning models were trained and evaluated on the processed loan default dataset.

A comparative analysis was performed using several evaluation metrics, enabling the selection of the most suitable model for credit risk prediction.

The trained model has been saved and is ready for explainability analysis and deployment.